# Analisis Kondisi Armada Operasional Damkar DKI Jakarta
## Exploratory Data Analysis (EDA) dan Klasifikasi Random Forest

**Proyek Akhir Data Science 2026/2027 — Kelompok 50 (Next Level)**  
Program Studi Informatika, Fakultas Teknik Industri, Universitas Pembangunan Nasional “Veteran” Yogyakarta.

**Anggota kelompok**
1. Dwirizki Adithya Putra — 123240021
2. Satya Dio Putranto — 123240121
3. Sebastian Alvaro Huller — 123240162
4. Paksi Dwirangga — 123240172

Notebook ini membahas data armada kendaraan pemadam kebakaran dan penyelamatan DKI Jakarta. Analisis dimulai dari pemahaman data, pemeriksaan pola, penyiapan fitur, hingga pengujian model klasifikasi. Random Forest digunakan sebagai model utama, sementara SVM dengan kernel linear dipakai untuk melihat perbandingan hasil.

> **Ruang lingkup:** kolom `Kondisi` berisi status yang tercatat pada dataset periode 2025. Hasil analisis tidak menggambarkan kesiapan kendaraan secara langsung pada saat ini dan tidak dapat menggantikan pemeriksaan teknis di lapangan.


## Daftar isi notebook

| Tahap | Pembahasan |
|---|---|
| 1. Business Understanding | Latar belakang, rumusan masalah, tujuan, metode, ukuran evaluasi, dan batasan |
| 2. Data Gathering | Asal data, proses pemuatan, penjelasan kolom, serta alasan pemilihan |
| 3. EDA | Struktur data, statistik ringkas, distribusi kondisi, dan pola berdasarkan karakteristik armada |
| 4. Cleaning & Preprocessing | Pemeriksaan kualitas, perapian nilai, penanganan data kosong, dan encoding |
| 5. Feature Selection & Modelling | Penentuan fitur, pembagian data, dan pembangunan model |
| 6. Evaluation | Perbandingan hasil, confusion matrix, dan feature importance |
| 7. Conclusion | Rangkuman temuan, makna hasil, saran, serta keterbatasan |

Analisis berfokus pada klasifikasi karena kolom `Kondisi` menyediakan dua label, yaitu `BAIK` dan `RUSAK`. EDA digunakan untuk memahami karakteristik dataset sebelum model dibangun. Sejumlah pendekatan lain tidak diterapkan karena tidak sesuai dengan tujuan analisis ini.


# BAGIAN 1 — BUSINESS UNDERSTANDING

## 1.1 Latar belakang dan masalah

Ketersediaan armada menjadi salah satu unsur penting dalam pelayanan pemadam kebakaran dan penyelamatan. Informasi mengenai jenis kendaraan, fungsi, wilayah penempatan, serta kondisi yang tercatat dapat membantu memberikan gambaran awal tentang komposisi armada.

Satu Data Jakarta menyediakan dataset kendaraan operasional yang memuat informasi tersebut. Data ini dapat ditelaah untuk melihat bagaimana kondisi kendaraan tersebar dan apakah karakteristik yang tersedia berkaitan dengan label kondisi. Perlu diperhatikan bahwa dataset merupakan catatan administratif untuk periode tertentu, bukan pemantauan kendaraan secara langsung.

## 1.2 Pertanyaan analisis

1. Bagaimana perbandingan jumlah kendaraan berlabel `BAIK` dan `RUSAK`?
2. Bagaimana sebaran kendaraan dan kondisi yang tercatat jika dilihat dari wilayah, jenis, klasifikasi, dan fungsi?
3. Seberapa baik fitur yang tersedia membantu model mengenali label kondisi pada data yang belum digunakan saat pelatihan?
4. Bagaimana perbedaan hasil Random Forest dan SVM kernel linear, khususnya dalam mengenali kelas `RUSAK`?
5. Fitur apa yang paling berpengaruh dalam pemisahan kelas pada model Random Forest?

## 1.3 Tujuan dan metode

Pekerjaan ini bertujuan menggambarkan karakteristik armada melalui EDA, menyiapkan data agar dapat dianalisis, dan menguji model untuk mengelompokkan catatan kendaraan ke dalam label `BAIK` atau `RUSAK`. Random Forest dipilih sebagai pendekatan utama, kemudian hasilnya dibandingkan dengan SVM kernel linear. Perbandingan dilakukan pada bagian data uji yang sama agar hasil kedua model dapat dibaca secara seimbang.

## 1.4 Metrik evaluasi

- **Accuracy:** persentase seluruh prediksi yang sesuai dengan label sebenarnya.
- **Precision kelas RUSAK:** bagian prediksi `RUSAK` yang memang berlabel `RUSAK`.
- **Recall kelas RUSAK:** bagian dari seluruh catatan `RUSAK` yang berhasil dikenali model.
- **F1-score kelas RUSAK:** ukuran yang menggabungkan precision dan recall untuk kelas tersebut.
- **Macro-F1:** rata-rata F1 dari kedua kelas dengan bobot yang sama, sehingga kelas dengan jumlah lebih sedikit tetap diperhitungkan.

## 1.5 Batasan awal

Dataset hanya menggambarkan periode 2025. Informasi seperti tanggal pemeriksaan, usia kendaraan, riwayat perawatan, jarak tempuh atau jam operasi, tingkat kerusakan, koordinat, dan catatan kejadian tidak tersedia pada berkas yang dianalisis. Oleh karena itu, hasil model hanya menunjukkan kemampuannya mengenali label dalam dataset, bukan menentukan penyebab kerusakan atau kondisi operasional kendaraan saat ini.


# BAGIAN 2 — DATA GATHERING

## 2.1 Sumber data

Data yang digunakan berasal dari portal resmi **Satu Data Jakarta**, dengan judul [Data Kendaraan Operasional Penanggulangan Kebakaran dan Penyelamatan](https://satudata.jakarta.go.id/open-data/detail?kategori=dataset&page_url=data-kendaraan-operasional-penanggulangan-kebakaran-dan-penyelamatan&data_no=1).

Pemeriksaan sumber dilakukan pada 20 November 2026. Salinan yang diperiksa memuat 1.153 baris dan 10 kolom. Berkas sumber berekstensi `.xls`, tetapi isinya terbaca sebagai tabel HTML sehingga pembacaan awal dilakukan menggunakan `pandas.read_html`.

**Lisensi:** Halaman umum Satu Data Jakarta menyatakan bahwa data pada portal dapat dimanfaatkan dan digunakan kembali. Lisensi dataset ini bersifat **Terbuka** dan telah berhasil diverifikasi melalui metadata dataset yang dapat diakses. Sumber data tetap dicantumkan sebagai bentuk atribusi. Berkas yang digunakan dalam analisis merupakan salinan sanitasi yang tidak menyertakan kolom `plat_nomor`.

## 2.2 Pemuatan dataset

Notebook membaca berkas CSV hasil sanitasi yang berada di folder `data/processed`. Kolom `plat_nomor` tidak disertakan karena merupakan pengenal kendaraan dan tidak dibutuhkan sebagai variabel untuk membangun model. Rangkuman pemeriksaan terhadap berkas sumber disimpan terpisah pada `data/source_audit.json`.


## 2.3 Gambaran dataset dan data dictionary

Setiap baris pada dataset menunjukkan satu catatan kendaraan dalam inventaris sumber. Kolom `plat_nomor` tidak disertakan pada berkas yang dipakai untuk analisis.

kolom di bawah disusun dengan melihat nama atribut dan nilai yang muncul di dalam data, Karena metadata resmi tidak menyediakan definisi yang lebih rinci.


## 2.4 Alasan pemilihan dataset

Dataset ini dipilih karena diterbitkan melalui portal data pemerintah daerah dan berkaitan langsung dengan layanan pemadam kebakaran serta penyelamatan. Jumlah datanya juga memadai untuk analisis, dengan 1.153 catatan dan sejumlah atribut yang dapat digunakan untuk menggambarkan karakteristik kendaraan. Selain itu, tersedia kolom `Kondisi` yang berisi label `BAIK` dan `RUSAK`, sehingga sesuai untuk percobaan klasifikasi.

Pada berkas sumber ditemukan lima nilai kosong di kolom `plat_nomor`. Kolom tersebut tidak dipakai dalam pemodelan karena hanya berfungsi sebagai pengenal kendaraan, sehingga dihilangkan dari salinan dataset yang dianalisis.


# BAGIAN 3 — EXPLORATORY DATA ANALYSIS (EDA)

## 3.1 Pemeriksaan struktur dan statistik deskriptif

Tahap awal dilakukan dengan memeriksa tipe data, jumlah nilai yang terisi, nilai unik, serta frekuensi setiap kategori. Pemeriksaan ini membantu mengenali bentuk dataset sebelum analisis lebih lanjut. Karena sebagian besar atribut berupa kategori, jumlah dan proporsi tiap kategori lebih mudah digunakan untuk membaca pola dibandingkan ukuran statistik seperti rata-rata atau standar deviasi.


## 3.2 Distribusi kondisi kendaraan

Bagian ini memperlihatkan jumlah dan persentase kendaraan berdasarkan label `BAIK` dan `RUSAK`. Perbandingan tersebut penting karena jumlah kedua kelas tidak seimbang. Kondisi ini perlu diperhatikan ketika membaca hasil model, sebab model dapat terlihat memiliki accuracy tinggi meskipun belum cukup baik mengenali kelas yang jumlahnya lebih sedikit.


**Interpretasi:** tabel dan grafik memperlihatkan bahwa catatan berlabel `BAIK` lebih banyak dibandingkan `RUSAK`. Ketimpangan jumlah ini membuat accuracy saja belum cukup untuk menilai model. Perhatian juga perlu diberikan pada recall dan F1-score kelas `RUSAK` agar terlihat seberapa banyak catatan pada kelas tersebut yang berhasil dikenali.


## 3.3 Komposisi armada menurut wilayah

Jumlah catatan kendaraan pada setiap wilayah ditampilkan untuk melihat sebaran armada. Grafik berikutnya membandingkan persentase kondisi di dalam masing-masing wilayah. Kedua tampilan ini perlu dibaca bersama karena jumlah kendaraan yang besar tidak selalu berarti proporsi kerusakannya juga tinggi.


**Interpretasi:** persentase `RUSAK` memberikan gambaran yang lebih sebanding antarwilayah daripada jumlah kasus semata. Wilayah dengan armada lebih banyak mungkin memiliki catatan `RUSAK` lebih banyak karena jumlah kendaraannya memang lebih besar. Hasil pada wilayah dengan jumlah catatan sedikit perlu dibaca secara hati-hati.


## 3.4 Komposisi menurut jenis, klasifikasi, dan fungsi

Bagian ini membandingkan jumlah kendaraan dan distribusi kondisi berdasarkan jenis kendaraan, klasifikasi, serta fungsi. Pengelompokan tersebut membantu melihat karakteristik armada yang paling banyak tercatat dan variasi label kondisi pada setiap kategori.


**Interpretasi:** tabel dan grafik dapat digunakan untuk melihat kategori yang memiliki persentase `RUSAK` relatif tinggi atau rendah. Namun, persentase perlu dilihat bersama jumlah kendaraan pada kategori tersebut. Perbedaan yang terlihat hanya menunjukkan pola dalam data dan tidak dapat dijadikan bukti bahwa jenis, fungsi, atau wilayah tertentu menjadi penyebab kerusakan.


## 3.5 Tiga temuan utama dari EDA

Selanjutnya dihitung tiga ringkasan yang mewakili pola penting dalam dataset. Angka ditarik langsung dari data agar uraian yang disampaikan sesuai dengan hasil perhitungan. Kategori dengan jumlah catatan kecil tetap perlu diperhatikan ketika menafsirkan persentasenya.


**Catatan:** temuan pada bagian ini menggambarkan keadaan yang tercatat di dalam dataset dan tidak menjelaskan hubungan sebab-akibat. Pada kelompok dengan jumlah kendaraan sedikit, perubahan satu atau dua catatan saja dapat mengubah persentase secara cukup besar.


# BAGIAN 4 — DATA CLEANING DAN PREPROCESSING

## 4.1 Pemeriksaan kualitas data awal

Pemeriksaan awal dilakukan untuk mengetahui nilai kosong, baris yang sama persis, dan kondisi setiap kolom. Pada salinan sumber lengkap terdapat lima nilai kosong di `plat_nomor`, sedangkan baris duplikat identik tidak ditemukan. Kolom `plat_nomor` kemudian dikeluarkan dari berkas analisis karena merupakan pengenal kendaraan.

Setelah kolom tersebut dihilangkan, terdapat 374 baris yang memiliki nilai sama pada sembilan kolom tersisa. Baris-baris ini tetap dipertahankan. Kesamaan atribut yang terlihat belum cukup untuk memastikan bahwa catatan tersebut berasal dari kendaraan yang sama; bisa saja kendaraan berbeda memiliki jenis, fungsi, lokasi, dan kondisi yang serupa.


## 4.2 Langkah pembersihan data

1. Menghapus spasi yang tidak diperlukan pada awal dan akhir teks.
2. Mengubah teks kosong menjadi nilai yang dikenali sebagai missing value.
3. Menyeragamkan penulisan label pada kolom `Kondisi` menjadi huruf kapital.
4. Mempertahankan baris dengan profil atribut yang sama setelah `plat_nomor` dihilangkan, karena identitas kendaraannya tidak lagi dapat dibedakan dari kolom yang tersisa.
5. Tidak mengisi nilai target yang kosong. Catatan tanpa label kondisi tidak dapat dipakai untuk melatih model terawasi. Nilai kosong pada fitur kategorikal ditangani menggunakan kategori yang paling sering muncul, dengan proses penghitungan dilakukan dari data latih.

Urutan pembersihan dan pengolahan ini menjaga agar informasi dari data uji tidak ikut digunakan ketika model mempelajari data.


## 4.3 Hasil pembersihan data

Jumlah baris tidak dikurangi hanya karena beberapa catatan tampak sama setelah proses sanitasi. Tanpa kolom pengenal, tidak ada dasar yang cukup untuk menyimpulkan bahwa catatan tersebut merupakan duplikasi kendaraan yang sama. Lima nilai kosong yang ditemukan pada sumber berada di kolom `plat_nomor`; kolom itu sudah dikeluarkan dari CSV analisis. Dengan demikian, tidak adanya missing value pada kolom CSV sanitasi tidak berarti berkas sumber awal bebas dari nilai kosong.


# BAGIAN 5 — FEATURE SELECTION DAN MODELLING

## 5.1 Pemilihan fitur dan target

Kolom `Kondisi` ditetapkan sebagai target yang akan diprediksi. `periode_data` tidak digunakan karena hanya berisi satu periode. Sementara itu, `kecamatan`, `kelurahan`, dan `penempatan` tidak dimasukkan ke model utama karena merupakan rincian lokasi yang memiliki banyak kategori dan saling bertingkat. Penggunaannya berpotensi membuat model terlalu bergantung pada lokasi tertentu.

Empat fitur yang digunakan adalah `wilayah`, `jenis_kendaraan`, `klasifikasi`, dan `fungsi`. Keempatnya dipilih untuk mewakili wilayah serta karakteristik kendaraan. Karena nilainya berbentuk kategori tanpa urutan, fitur diubah menggunakan One-Hot Encoding agar setiap kategori direpresentasikan sebagai kolom biner.


## 5.2 Pembagian data latih dan data uji

Dataset dibagi menjadi 80% data latih dan 20% data uji. Nilai `random_state=42` digunakan agar pembagian dapat diulang, sedangkan `stratify=y` menjaga agar perbandingan kelas `BAIK` dan `RUSAK` relatif sama pada kedua bagian. Data uji disimpan untuk mengukur kemampuan model pada catatan yang tidak ikut digunakan selama proses pelatihan.


## 5.3 Pengolahan fitur dan model SVM

SVM digunakan sebagai model pembanding. Sebelum pelatihan, nilai kosong pada fitur kategorikal ditangani dengan modus, kemudian setiap kategori diubah menjadi representasi One-Hot. Hasil encoding selanjutnya distandarisasi menggunakan `StandardScaler(with_mean=False)` agar format sparse tetap dapat digunakan. Seluruh tahapan dirangkai dalam `Pipeline`, sehingga proses pengolahan dipelajari dari data latih dan diterapkan dengan cara yang sama pada data uji.


## 5.4 Model utama Random Forest

Random Forest dibangun menggunakan kumpulan pohon keputusan. Beberapa pengaturan yang diuji mencakup jumlah pohon (`n_estimators`) dan kedalaman maksimum (`max_depth`). Konfigurasi akhir menggunakan 100 pohon, kedalaman maksimum 10, kriteria pemisahan Gini, dan `random_state=42`.

Jumlah label `BAIK` dan `RUSAK` tidak seimbang. Untuk mengurangi kecenderungan model mengutamakan kelas mayoritas, konfigurasi akhir memakai `class_weight='balanced'`, sehingga kelas dengan jumlah lebih sedikit memperoleh bobot lebih besar saat proses pembelajaran. Fitur kategorikal tetap diubah menjadi One-Hot, tetapi tidak memerlukan scaling seperti pada SVM karena pemisahan pada pohon keputusan tidak didasarkan pada jarak antarobservasi.


# BAGIAN 6 — EVALUATION

## 6.1 Perbandingan kinerja model

Random Forest dan SVM dinilai menggunakan bagian data uji yang sama. Accuracy ditampilkan bersama macro-F1, precision, recall, dan F1-score untuk kelas `RUSAK`. Dengan melihat beberapa ukuran sekaligus, penilaian tidak hanya bergantung pada jumlah prediksi benar secara keseluruhan, tetapi juga memperhatikan kemampuan model mengenali kelas yang lebih sedikit.


## 6.2 Confusion matrix

Pada confusion matrix, baris menunjukkan label sebenarnya, sedangkan kolom menunjukkan label hasil prediksi. Perhatian khusus diberikan pada catatan yang sebenarnya berlabel `RUSAK` tetapi diprediksi sebagai `BAIK`. Kesalahan ini disebut false negative untuk kelas `RUSAK`, yaitu catatan rusak yang tidak berhasil dikenali model.


## 6.3 Menentukan model yang lebih sesuai

Perbandingan model mempertimbangkan macro-F1 dan recall kelas `RUSAK`, selain accuracy. Macro-F1 memperlakukan kedua kelas secara setara, sedangkan recall `RUSAK` menunjukkan seberapa banyak catatan rusak yang berhasil ditemukan. Jika nilai beberapa metrik berdekatan, hasil perlu ditafsirkan dengan mempertimbangkan jumlah data dan fakta bahwa evaluasi dilakukan melalui satu kali pembagian data latih-uji.


## 6.4 Feature importance Random Forest

Random Forest menyediakan nilai `feature_importances_` untuk menunjukkan kontribusi relatif fitur dalam proses pemisahan kelas pada pohon-pohon yang dibentuk. Karena fitur kategorikal telah diubah menjadi beberapa kolom One-Hot, nilai dari kategori-kategori yang berasal dari satu atribut dijumlahkan kembali. Dengan cara ini, tingkat kepentingan setiap fitur asal dapat dibandingkan dengan lebih mudah.


**Interpretasi feature importance:** nilai yang lebih besar menunjukkan bahwa fitur tersebut lebih banyak berperan dalam pemisahan kelas pada model Random Forest. Angka ini tidak membuktikan hubungan sebab-akibat dan tidak berarti fitur tersebut menjadi penyebab kondisi kendaraan. Banyaknya kategori serta keterkaitan antarfitur juga dapat memengaruhi pembagian nilai importance.


# BAGIAN 7 — CONCLUSION

## 7.1 Jawaban atas pertanyaan analisis

Rangkuman pada bagian ini disusun dari hasil perhitungan yang diperoleh ketika notebook dijalankan. Dengan demikian, angka dan temuan yang ditampilkan merujuk pada dataset serta hasil evaluasi model dalam proyek ini.


## 7.2 Hasil

- Perbandingan label `BAIK` dan `RUSAK` menunjukkan kondisi yang tercatat pada data periode 2025, bukan keadaan kendaraan secara langsung saat ini.
- Recall `RUSAK` menggambarkan bagian catatan berlabel `RUSAK` yang berhasil ditemukan model. Nilai yang rendah berarti masih banyak catatan pada kelas tersebut yang diprediksi sebagai `BAIK`.
- Precision `RUSAK` menunjukkan ketepatan prediksi model ketika memberikan label `RUSAK`. Jika nilainya rendah, sebagian prediksi tersebut ternyata memiliki label `BAIK` pada data uji.
- Model sebaiknya digunakan untuk membantu memahami pola dalam data, bukan sebagai dasar tunggal untuk menentukan penugasan atau kondisi teknis kendaraan.




## 7.3 Keterbatasan

1. Data yang dianalisis hanya mencakup satu periode, yaitu tahun 2025, sehingga perubahan kondisi dari waktu ke waktu belum dapat diamati.
2. Berkas yang digunakan tidak menjelaskan secara rinci kriteria penetapan label `BAIK` dan `RUSAK`, tanggal pemeriksaan, maupun prosedur inspeksinya.
3. Kolom pengenal kendaraan dihilangkan dari berkas analisis. Karena itu, sejumlah baris dengan atribut yang sama tidak dapat dipastikan sebagai kendaraan yang sama atau berbeda.
4. Fitur model hanya mencakup wilayah, jenis kendaraan, klasifikasi, dan fungsi. Informasi usia, intensitas penggunaan, perawatan, serta tingkat kerusakan belum tersedia.
5. Evaluasi dilakukan dengan satu pembagian data latih dan data uji. Pengujian pada data dari periode lain masih diperlukan untuk mengetahui apakah hasilnya dapat diterapkan secara lebih luas.
6. Lisensi dataset bersifat **Terbuka** dan telah berhasil diverifikasi melalui metadata dataset yang dapat diakses. Halaman umum Satu Data Jakarta menyatakan bahwa data pada portal dapat dimanfaatkan dan digunakan kembali. Atribusi sumber tetap dicantumkan.
7. Hasil klasifikasi hanya berkaitan dengan label pada dataset dan tidak menjelaskan penyebab kerusakan maupun kesiapan kendaraan secara real-time.

## 7.4 Kesimpulan akhir

Analisis ini memberikan gambaran mengenai data kendaraan operasional pemadam kebakaran dan penyelamatan DKI Jakarta periode 2025 melalui EDA dan klasifikasi. Random Forest digunakan sebagai model utama, sedangkan SVM kernel linear menjadi pembanding. Penilaian dilakukan dengan melihat confusion matrix, classification report, macro-F1, serta precision dan recall untuk kelas `RUSAK`.

Hasil pengujian menunjukkan bahwa setiap model memiliki keunggulan pada ukuran evaluasi yang berbeda. Oleh karena itu, pemilihan model perlu disesuaikan dengan tujuan penggunaan dan tidak cukup didasarkan pada accuracy saja. Seluruh temuan tetap terbatas pada informasi yang tersedia di dataset dan tidak menggantikan pemeriksaan langsung terhadap kendaraan.
